# Bootstrapping

This notebook implements both a traditional bootstrap and a block bootstrap in order to get more robust standard errors of OLS coefficients.

## Load Packages and Extra Functions

The key functions (for doing OLS and block bootstrap) are from the (local) `FinEcmt_OLS` module.

In [1]:
MyModulePath = joinpath(pwd(),"src")
!in(MyModulePath,LOAD_PATH) && push!(LOAD_PATH,MyModulePath)
using FinEcmt_OLS

In [2]:
#=
include(joinpath(pwd(),"src","FinEcmt_OLS.jl"))
using .FinEcmt_OLS
=#

In [3]:
using DelimitedFiles, Statistics, LinearAlgebra, Random

## Loading Data

The data are for 

In [4]:
x = readdlm("Data/TwoIndustries.csv",',',skipstart=1)

(dN,Re,F) = (x[:,1],Float64.(x[:,2:3]),Float64.(x[:,4:end]))   #make sure data is Float
x = nothing
printlnPs("Re: ",size(Re),"\n","F: ",size(F))

y = Re[:,1]
T = length(y)
x = [ones(T) F[:,1]]           #constant, market excess return
k = size(x,2)

println("\nT and k: $T $k")

      Re:   (660, 2)          
       F:   (660, 3)

T and k: 660 2


## Point Estimates and Standard Errors

In [5]:
(bLS,u,yhat,Covb,) = OlsGM(y,x)            #OLS estimate and traditional std errors
t_iid = bLS./sqrt.(diag(Covb))

CovW = OlsNW(y,x,0)[4]                    #White's VCV
t_W  = bLS./sqrt.(diag(CovW))
CovNW = OlsNW(y,x,2)[4]                   #Newey-West VCV  
t_NW  = bLS./sqrt.(diag(CovNW))

printblue("OLS estimates:\n")
rowNames = [string("x",'₀'+i) for i=1:k]      #'₀'+1 to get ₁
printmat(bLS,t_iid,t_W,t_NW;colNames=["coeff","t (iid)","t (White)","t (NW, 2)"],rowNames=rowNames,width=15)

OLS estimates:

            coeff        t (iid)      t (White)      t (NW, 2)
x₁         -0.052         -0.412         -0.412         -0.414
x₂          1.237         45.301         39.016         32.944



## Standard Bootstrap (1)

In each loop, a new series of residuals, $\tilde{u}_{t}$, is created by drawing (with replacement) values from the fitted residuals (from the estimates in earlier cells). Then, simulated values of the dependent variable are created as 

$\tilde{y}_{t}=x_{t}^{\prime}\beta+\tilde{u}_{t}$ 

and we redo the estimation on ($\tilde{y}_{t},x_{t}$). Notice that $x_t$ is the same as in the data.

This is repeated `NSim` times.

In [6]:
@doc2 OlsBootstrap
#using CodeTracking
#println(@code_string OlsBootstrap([],[],[],1,1))    #print the source code

```julia
OlsBootstrap(y,x,bLS,BlockSize,NSim,ExciseIt=false;yxPairType=0)
```

Do a bootstrap of OLS regression, w/wo blocks and of residual/pairs for (y,x)

# Input

  * `y::Array`:            Txn matrix of n dependent variables
  * `x::Array`:            TxK matrix of (common) regressors
  * `bLS::Array`:          OLS estimates (will be generated if not supplied)
  * `BlockSize::Int`:      scalar, size of blocks for block bootstrap                        (eg. 1 for no blocks, 20 for long blocks)
  * `NSim::Int`:           scalar, number of simulations
  * `ExciseIt::Bool`:      true for excise on [y,x]; default: false
  * `yxPairType::Int`:     1 for drawing pairs of (y(s),x(s)); 2 for wild bootstrap; else residuals are drawn; default: 0

# Output

  * `CovbBoot::Array`:     K*n x K*n covariance matrix of vec(b)
  * `bBoot::Array`:        NSim x K*n matrix, vec(b) in row i

# Requires

  * FindNN


In [7]:
NSim      = 3000                 #no. of simulations
Random.seed!(123)

(CovbBoot,bBoot) = OlsBootstrap(y,x,[],1,NSim)

printblue("Coefficients:")
xx = [bLS  mean(bBoot,dims=1)']
printmat(xx;colNames=["OLS","avg. bootstr"],rowNames=rowNames,width=20)

printblue("t-stats:")
xx = [t_iid bLS./sqrt.(diag(CovbBoot))]
printmat(xx;colNames=["t (iid)","t (bootstrap 1)"],rowNames=rowNames,width=20)

printred("The results from these bootstrap are similar to standard OLS results, cf. above")

Coefficients:
                   OLS        avg. bootstr
x₁              -0.052              -0.047
x₂               1.237               1.237

t-stats:
               t (iid)     t (bootstrap 1)
x₁              -0.412              -0.416
x₂              45.301              45.772

The results from these bootstrap are similar to standard OLS results, cf. above


## Pairwise Bootstap (2)

to handle heteroskedasticity.

In [8]:
CovbBoot, = OlsBootstrap(y,x,[],1,NSim;yxPairType=1)    #pairwise

printblue("t-stats:")
xx = [t_W  bLS./sqrt.(diag(CovbBoot))]
printmat(xx;colNames=["t (White's)","t (bootstrap 2)"],rowNames=rowNames,width=20)

printred("The results from these bootstrap are similar to White's standard error, cf. above")

t-stats:
           t (White's)     t (bootstrap 2)
x₁              -0.412              -0.414
x₂              39.016              39.116

The results from these bootstrap are similar to White's standard error, cf. above


## Block Bootstrap & Pairwise (3)

To handle autocorrelations and heteroskedasticity, we do a block bootstrap of the $(y_t,x_t)$ pairs.

In [9]:
CovbBoot, = OlsBootstrap(y,x,[],5,NSim;yxPairType=1)    #pairwise, blocks of 5

printblue("t-stats:")
xx = [t_NW  bLS./sqrt.(diag(CovbBoot))]
printmat(xx;colNames=["t (NW, 2)","t (bootstrap 3)"],rowNames=rowNames,width=20)

printred("The results from these bootstrap are similar to NW's standard error, cf. above")

t-stats:
             t (NW, 2)     t (bootstrap 3)
x₁              -0.414              -0.403
x₂              32.944              29.267

The results from these bootstrap are similar to NW's standard error, cf. above
